In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Labs"
)

xpt_file = DATA_DIR / "FASTQX_G.xpt"
csv_file = DATA_DIR / "FASTQX_G_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 80)
print("NHANES 2011-2012 FASTQX_G XPT -> CSV")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "PHQ020",
    "PHACOFHR",
    "PHACOFMN",
    "PHQ030",
    "PHAALCHR",
    "PHAALCMN",
    "PHQ040",
    "PHAGUMHR",
    "PHAGUMMN",
    "PHQ050",
    "PHAANTHR",
    "PHAANTMN",
    "PHQ060",
    "PHASUPHR",
    "PHASUPMN",
    "PHAFSTHR",
    "PHAFSTMN",
    "PHDSESN"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "FASTQX_G column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 8956:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 19:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 8,956 rows x 19 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# IMPORTANT:
#
# NEVER globally replace this value without validating
# the exact variables and counts first.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


# ============================================================
# 5. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact pattern found in attached FASTQX_G.xpt:
#
# PHDSESN     4233
# PHAFSTHR     699
# PHAFSTMN     148
# PHAGUMHR     119
# PHAGUMMN      12
# PHACOFHR       6
# PHAANTHR       3
# PHASUPHR       3
# PHASUPMN       2
# PHAALCHR       1
# PHAALCMN       1
# PHACOFMN       1
#
# TOTAL = 5,228
#
#
# CDC validates zero as legitimate for these fields:
#
# PHACOFHR    range 0-14 hours
# PHACOFMN    range 0-59 minutes
#
# PHAALCHR    range 0-33 hours
# PHAALCMN    range 0-59 minutes
#
# PHAGUMHR    range 0-16 hours
# PHAGUMMN    range 0-59 minutes
#
# PHAANTHR    range 0-18 hours
#
# PHASUPHR    range 0-24 hours
# PHASUPMN    range 0-59 minutes
#
# PHAFSTHR    range 0-36 hours
# PHAFSTMN    range 0-59 minutes
#
# PHDSESN
#     0 = Morning
#

EXPECTED_TINY_COUNTS = {

    "PHACOFHR": 6,

    "PHACOFMN": 1,

    "PHAALCHR": 1,

    "PHAALCMN": 1,

    "PHAGUMHR": 119,

    "PHAGUMMN": 12,

    "PHAANTHR": 3,

    "PHASUPHR": 3,

    "PHASUPMN": 2,

    "PHAFSTHR": 699,

    "PHAFSTMN": 148,

    "PHDSESN": 4233
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated FASTQX_G pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 5228:

    raise ValueError(
        f"Unexpected total tiny count: {tiny_total:,}"
    )


if len(EXPECTED_TINY_COUNTS) != 12:

    raise ValueError(
        "Expected tiny values in exactly 12 variables."
    )


print(
    "\nPASS: Exact 5,228-value tiny-zero "
    "pattern across 12 variables validated."
)


# ============================================================
# 6. RESTORE ONLY VALIDATED ZERO VALUES
# ============================================================

#
# Do NOT perform a global:
#
#     df.replace(TINY_VALUE, 0)
#
# Only restore values in variables whose exact pattern
# was validated above.
#

print("\n--- ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    tiny_mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        tiny_mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        tiny_mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 5228:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zeros restored."
)


# ============================================================
# 7. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# XPT should contain.
#
# The ONLY numeric changes permitted are:
#
#     validated TINY_VALUE -> 0
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col].eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 8. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining after correction:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 9. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# IMPORTANT:
#
# Check BEFORE converting anything to Int64.
#
# FASTQX_G measures fasting time as separate
# whole-number:
#
#     hours
#     minutes
#
# It therefore has NO genuine fractional fields.
#
# If fractions unexpectedly appear, STOP rather than
# rounding them.
#
# This is the same safeguard used to protect genuine
# Dietary, Laboratory and Examination decimals.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )


        print(
            f"{col}: "
            f"{int(fractional_mask.sum()):,} "
            "fractional observations"
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine fractional values detected. "
        "They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables "
    "exist in FASTQX_G."
)


# ============================================================
# 10. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if not (
    df["SEQN"].min() == 62161
    and
    df["SEQN"].max() == 71916
    and
    int(
        df["SEQN"]
        .isna()
        .sum()
    ) == 0
    and
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 11. FORMAT ALL PROVEN WHOLE-NUMBER FIELDS
# ============================================================

#
# After:
#
#     tiny-zero restoration
#     fractional-value validation
#
# all 19 FASTQX_G variables are proven whole-number.
#
# Convert to nullable Int64.
#
# IMPORTANT:
#
# NO .round() IS USED.
#

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 19 FASTQX_G fields "
    "converted to nullable Int64."
)


# ============================================================
# 12. HELPER - CDC FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs from CDC.\n"
            f"Expected: {expected_missing}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 13. CDC CHECK - COFFEE / TEA
# ============================================================

#
# PHQ020:
#
# 1 = Yes     71
# 2 = No    8457
# Missing    428
#

check_frequency(
    "PHQ020",
    {
        1: 71,
        2: 8457
    },
    428
)


#
# PHACOFHR:
#
# range 0-14
# n = 71
# missing = 8,885
# zero = 6
#

s = df["PHACOFHR"]


if not (
    s.min() == 0
    and
    s.max() == 14
    and
    int(
        s.notna()
        .sum()
    ) == 71
    and
    int(
        s.isna()
        .sum()
    ) == 8885
    and
    int(
        s.eq(0)
        .sum()
    ) == 6
):

    raise ValueError(
        "PHACOFHR differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: PHACOFHR matches CDC."
)


#
# PHACOFMN:
#
# range 0-59
# n = 71
# missing = 8,885
# zero = 1
#

s = df["PHACOFMN"]


if not (
    s.min() == 0
    and
    s.max() == 59
    and
    int(
        s.notna()
        .sum()
    ) == 71
    and
    int(
        s.isna()
        .sum()
    ) == 8885
    and
    int(
        s.eq(0)
        .sum()
    ) == 1
):

    raise ValueError(
        "PHACOFMN differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: PHACOFMN matches CDC."
)


# ============================================================
# 14. CDC CHECK - ALCOHOL
# ============================================================

check_frequency(
    "PHQ030",
    {
        1: 36,
        2: 8492
    },
    428
)


#
# PHAALCHR:
#
# range 0-33
# n = 36
# zero = 1
#

s = df["PHAALCHR"]


if not (
    s.min() == 0
    and
    s.max() == 33
    and
    int(
        s.notna()
        .sum()
    ) == 36
    and
    int(
        s.isna()
        .sum()
    ) == 8920
    and
    int(
        s.eq(0)
        .sum()
    ) == 1
):

    raise ValueError(
        "PHAALCHR differs from CDC."
    )


print(
    "PASS: PHAALCHR matches CDC."
)


#
# PHAALCMN:
#
# range 0-59
# n = 36
# zero = 1
#

s = df["PHAALCMN"]


if not (
    s.min() == 0
    and
    s.max() == 59
    and
    int(
        s.notna()
        .sum()
    ) == 36
    and
    int(
        s.isna()
        .sum()
    ) == 8920
    and
    int(
        s.eq(0)
        .sum()
    ) == 1
):

    raise ValueError(
        "PHAALCMN differs from CDC."
    )


print(
    "PASS: PHAALCMN matches CDC."
)


# ============================================================
# 15. CDC CHECK - GUM / MINTS / COUGH DROPS
# ============================================================

check_frequency(
    "PHQ040",
    {
        1: 277,
        2: 8251
    },
    428
)


#
# PHAGUMHR
#
# range 0-16
# n = 277
# zero = 119
#

s = df["PHAGUMHR"]


if not (
    s.min() == 0
    and
    s.max() == 16
    and
    int(
        s.notna()
        .sum()
    ) == 277
    and
    int(
        s.isna()
        .sum()
    ) == 8679
    and
    int(
        s.eq(0)
        .sum()
    ) == 119
):

    raise ValueError(
        "PHAGUMHR differs from CDC."
    )


print(
    "PASS: PHAGUMHR matches CDC."
)


#
# PHAGUMMN
#
# range 0-59
# zero = 12
#

s = df["PHAGUMMN"]


if not (
    s.min() == 0
    and
    s.max() == 59
    and
    int(
        s.notna()
        .sum()
    ) == 277
    and
    int(
        s.isna()
        .sum()
    ) == 8679
    and
    int(
        s.eq(0)
        .sum()
    ) == 12
):

    raise ValueError(
        "PHAGUMMN differs from CDC."
    )


print(
    "PASS: PHAGUMMN matches CDC."
)


# ============================================================
# 16. CDC CHECK - ANTACIDS / LAXATIVES
# ============================================================

check_frequency(
    "PHQ050",
    {
        1: 23,
        2: 8505
    },
    428
)


#
# PHAANTHR:
#
# range 0-18
# n = 23
# zero = 3
#

s = df["PHAANTHR"]


if not (
    s.min() == 0
    and
    s.max() == 18
    and
    int(
        s.notna()
        .sum()
    ) == 23
    and
    int(
        s.isna()
        .sum()
    ) == 8933
    and
    int(
        s.eq(0)
        .sum()
    ) == 3
):

    raise ValueError(
        "PHAANTHR differs from CDC."
    )


print(
    "PASS: PHAANTHR matches CDC."
)


#
# PHAANTMN is important because:
#
# CDC range = 5-53
#
# It does NOT legitimately contain zero in this file.
# It therefore did NOT appear in the tiny-value pattern.
#

s = df["PHAANTMN"]


if not (
    s.min() == 5
    and
    s.max() == 53
    and
    int(
        s.notna()
        .sum()
    ) == 23
    and
    int(
        s.isna()
        .sum()
    ) == 8933
    and
    int(
        s.eq(0)
        .sum()
    ) == 0
):

    raise ValueError(
        "PHAANTMN differs from CDC."
    )


print(
    "PASS: PHAANTMN matches CDC."
)


# ============================================================
# 17. CDC CHECK - DIETARY SUPPLEMENTS
# ============================================================

check_frequency(
    "PHQ060",
    {
        1: 71,
        2: 8457
    },
    428
)


#
# PHASUPHR:
#
# 0-24
# n = 71
# zero = 3
#

s = df["PHASUPHR"]


if not (
    s.min() == 0
    and
    s.max() == 24
    and
    int(
        s.notna()
        .sum()
    ) == 71
    and
    int(
        s.isna()
        .sum()
    ) == 8885
    and
    int(
        s.eq(0)
        .sum()
    ) == 3
):

    raise ValueError(
        "PHASUPHR differs from CDC."
    )


print(
    "PASS: PHASUPHR matches CDC."
)


#
# PHASUPMN:
#
# 0-59
# n = 71
# zero = 2
#

s = df["PHASUPMN"]


if not (
    s.min() == 0
    and
    s.max() == 59
    and
    int(
        s.notna()
        .sum()
    ) == 71
    and
    int(
        s.isna()
        .sum()
    ) == 8885
    and
    int(
        s.eq(0)
        .sum()
    ) == 2
):

    raise ValueError(
        "PHASUPMN differs from CDC."
    )


print(
    "PASS: PHASUPMN matches CDC."
)


# ============================================================
# 18. CDC CHECK - TOTAL FASTING TIME
# ============================================================

#
# PHAFSTHR:
#
# CDC:
#
# 0-36 hours = 8,528 observations
# missing    =   428
#
# Attached XPT:
#
# zero = 699
#

s = df["PHAFSTHR"]


print("\n--- PHAFSTHR ---")


print(
    "Zeros:",
    int(
        s.eq(0)
        .sum()
    )
)


if not (
    s.min() == 0
    and
    s.max() == 36
    and
    int(
        s.notna()
        .sum()
    ) == 8528
    and
    int(
        s.isna()
        .sum()
    ) == 428
    and
    int(
        s.eq(0)
        .sum()
    ) == 699
):

    raise ValueError(
        "PHAFSTHR differs from CDC / "
        "zero restoration."
    )


print(
    "PASS: PHAFSTHR matches CDC."
)


#
# PHAFSTMN:
#
# CDC:
#
# 0-59 minutes = 8,528
# missing      =   428
#
# zero = 148
#

s = df["PHAFSTMN"]


print("\n--- PHAFSTMN ---")


print(
    "Zeros:",
    int(
        s.eq(0)
        .sum()
    )
)


if not (
    s.min() == 0
    and
    s.max() == 59
    and
    int(
        s.notna()
        .sum()
    ) == 8528
    and
    int(
        s.isna()
        .sum()
    ) == 428
    and
    int(
        s.eq(0)
        .sum()
    ) == 148
):

    raise ValueError(
        "PHAFSTMN differs from CDC / "
        "zero restoration."
    )


print(
    "PASS: PHAFSTMN matches CDC."
)


# ============================================================
# 19. CDC CHECK - EXAM SESSION
# ============================================================

#
# CDC:
#
# PHDSESN
#
# 0 = Morning       4,233
# 1 = Afternoon     3,294
# 2 = Evening       1,429
#
# no missing
#
# This is the largest XPORT zero-restoration field.
#

EXPECTED_PHDSESN = {
    0: 4233,
    1: 3294,
    2: 1429
}


actual = (
    df["PHDSESN"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual != EXPECTED_PHDSESN:

    raise ValueError(
        "PHDSESN differs from CDC.\n"
        f"Expected: {EXPECTED_PHDSESN}\n"
        f"Actual:   {actual}"
    )


if int(
    df["PHDSESN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "PHDSESN contains unexpected missing values."
    )


print(
    "PASS: PHDSESN matches CDC exactly."
)


# ============================================================
# 20. VALIDATED ZERO COUNT CHECK
# ============================================================

#
# Since this attached XPT originally contained zero
# ordinary numeric zeros, each post-correction zero count
# should equal its validated tiny-artifact count.
#

print("\n--- VALIDATED ZERO COUNTS ---")


for col, expected_zero_count in EXPECTED_TINY_COUNTS.items():

    actual_zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    print(
        f"{col}: "
        f"{actual_zero_count:,}"
    )


    if actual_zero_count != expected_zero_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_zero_count:,} zeros, "
            f"found {actual_zero_count:,}."
        )


print(
    "PASS: All validated zero counts match."
)


# ============================================================
# 21. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# The ONLY expected numeric differences from raw pandas
# XPT are the 5,228 validated:
#
#     TINY_VALUE -> 0
#
# restorations.
#
# Nothing else is allowed to change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 19 variables match the "
    "expected corrected XPT exactly."
)


# ============================================================
# 22. EXPORT CSV
# ============================================================

#
# No:
#
#     float_format=
#
# No:
#
#     rounding
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 23. RE-READ EXPORTED CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if csv_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    csv_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 24. EXACT CORRECTED XPT -> CSV FIDELITY
# ============================================================

print("\n--- EXACT XPT -> CSV FIDELITY ---")


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        csv_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 19 CSV columns round-trip "
    "to exact corrected XPT numeric values."
)


# ============================================================
# 25. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 26. READ RAW CSV AS TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 27. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 28. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# FASTQX_G contains NO genuine decimal variables.
#
# Therefore ".0" should not occur anywhere in this
# particular CSV.
#
# IMPORTANT:
#
# This is NOT the rule for Dietary, continuous Lab,
# or Examination datasets containing real decimals.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected .0 formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary .0 formatting."
)


# ============================================================
# 29. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 30. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if not (
    final_seqn.min() == 62161
    and
    final_seqn.max() == 71916
    and
    int(
        final_seqn.duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains .0."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no .0."
)


# ============================================================
# 31. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 80
)


print(
    "FASTQX_G CONVERSION COMPLETE"
)


print(
    "=" * 80
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Variables containing tiny artifacts:",
    len(EXPECTED_TINY_COUNTS)
)


print(
    "Total validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "Whole-number variables:",
    len(INTEGER_COLUMNS)
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary .0: "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2011-2012 FASTQX_G XPT -> CSV
Rows: 8,956
Columns: 19
PASS: Structure = 8,956 rows x 19 columns.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros: 0
Tiny artifact occurrences: 5,228

Tiny values by variable:
PHDSESN     4233
PHAFSTHR     699
PHAFSTMN     148
PHAGUMHR     119
PHAGUMMN      12
PHACOFHR       6
PHASUPHR       3
PHAANTHR       3
PHASUPMN       2
PHAALCMN       1
PHACOFMN       1
PHAALCHR       1

PASS: Exact 5,228-value tiny-zero pattern across 12 variables validated.

--- ZERO RESTORATION ---
PHACOFHR: 6 restored to 0
PHACOFMN: 1 restored to 0
PHAALCHR: 1 restored to 0
PHAALCMN: 1 restored to 0
PHAGUMHR: 119 restored to 0
PHAGUMMN: 12 restored to 0
PHAANTHR: 3 restored to 0
PHASUPHR: 3 restored to 0
PHASUPMN: 2 restored to 0
PHAFSTHR: 699 restored to 0
PHAFSTMN: 148 restored to 0
PHDSESN: 4,233 restored to 0

PASS: 5,228 validated zeros restored.

Tiny values remaining after correction: 0
PASS: No tiny XPORT artifacts remain.

--- FRACTIONAL VALUE CHECK ---
